# Deep Double Descent — Where Bigger Models and More Data Hurt (2019)

_Papers · Foundations & Optimization_

**As model size grows, test error first falls, then peaks right where the model just fits the training set, then falls again.**

---

This notebook is a practice scaffold for the **Deep Double Descent — Where Bigger Models and More Data Hurt (2019)** lesson. We reproduce the double-descent curve one piece at a time: run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Sanity-check ordinary least squares by hand

Before sweeping model size, fit a tiny under-parameterized model so the regime is concrete. With `D = 2` features (bias + x) and `n = 3` points, the model has fewer knobs than data points, so it **cannot** pass through every point — the training error is non-zero. We solve the normal equations directly to see this.

In [ ]:
import numpy as np
import math

# Sanity-check the worked example: OLS fit, D=2 features, n=3 points.
Phi = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]])   # bias + x, for x in {0,1,2}
yw  = np.array([1.0, 1.0, 3.0])                          # noisy targets

theta = np.linalg.solve(Phi.T @ Phi, Phi.T @ yw)        # normal equations
pred  = Phi @ theta
train_mse = np.mean((pred - yw)**2)

print("worked example: theta=(%.4f, %.4f)  trainMSE=%.4f" % (
      theta[0], theta[1], train_mse))
# worked example: theta=(0.6667, 1.0000)  trainMSE=0.2222
# D=2 < n=3 : under-parameterized, cannot interpolate, error is non-zero.

### Step 2 — Make the data and a bank of random features

We draw high-dimensional Gaussian inputs and a fixed nonlinear teacher signal. Critically, the **train** labels carry noise while the **test** labels are clean — interpolating the noisy train labels is exactly what produces the peak. We also pre-build one large bank of random ReLU features; taking the first `D` columns gives us a model of size `D`, with the interpolation threshold sitting at `D = n_train = 60`.

In [ ]:
# Data: high-dim Gaussian inputs; NOISY train labels, CLEAN test labels.
np.random.seed(0)
d, n_train, n_test, noise_std = 20, 60, 4000, 0.5     # interpolation threshold at D = n_train = 60

Xtr = np.random.randn(n_train, d)
Xte = np.random.randn(n_test,  d)
beta = np.random.randn(d) / math.sqrt(d)

def teacher(X):
    return np.tanh(X @ beta) + 0.3 * (X @ beta)        # fixed nonlinear signal

ytr = teacher(Xtr) + noise_std * np.random.randn(n_train)     # train labels are NOISY
yte = teacher(Xte)                                            # test labels are CLEAN

# A fixed bank of random ReLU features; first D columns = a size-D model.
Dmax  = 600
Wbank = np.random.randn(Dmax, d) / math.sqrt(d)

def feats(X, D):
    return np.maximum(X @ Wbank[:D].T, 0.0)            # phi(x) = ReLU(W x)

### Step 3 — Sweep model size and watch the test error peak

Now grow `D` from tiny to far past `n = 60`. For each size we fit with the pseudo-inverse, which gives ordinary least squares when `D < n` and the **minimum-norm** exact fit when `D ≥ n`. Watch the train error fall to zero exactly at `D = 60` and stay there, while the test error spikes at `D = 60` then descends again — the double descent.

In [ ]:
# Sweep model size D. Min-norm least squares via the pseudo-inverse.
#     pinv gives ordinary regression when D<n and the MINIMUM-NORM exact fit when D>=n.
Ds = [2,5,10,15,20,30,40,48,54,58,60,62,66,72,84,100,130,180,250,350,500,600]

print("\n   D   trainMSE     testMSE")
for D in Ds:
    Ptr = feats(Xtr, D)
    Pte = feats(Xte, D)
    theta = np.linalg.pinv(Ptr) @ ytr
    tr = np.mean((Ptr @ theta - ytr)**2)
    te = np.mean((Pte @ theta - yte)**2)
    mark = "  <-- interpolation threshold D = n" if D == n_train else ""
    print("%4d   %8.4f   %9.4f%s" % (D, tr, te, mark))

#    D   trainMSE     testMSE
#    2     0.9437      0.7990
#   20     0.3883      0.7561   (classical descent)
#   54     0.0897      3.3675   (rising toward the peak)
#   60     0.0000     48.8568   <-- interpolation threshold D = n  (TEST ERROR PEAK)
#   66     0.0000      3.0518   (second descent begins)
#  600     0.0000      0.2183   (over-parameterized: BEST of all)
# Train error -> 0 exactly at D=60=n and stays 0. Test error PEAKS at D=60, then falls
# again to 0.218 -- far below the classical-regime best (~0.72). That is double descent.
# (Our small run, not the paper's reported numbers. Values vary by seed/hardware.)

## Visualize the data & results

_As we grow the model size D (number of random features) past the number of training samples n=60, what happens to training error and test error?_

### Step 4 — Rebuild the sweep self-contained

This visualization cell is standalone, so it re-imports NumPy and re-creates the data and the random-feature bank exactly as above. Keeping it self-contained means the panel reproduces even if run in isolation.

In [ ]:
import numpy as np
import math

np.random.seed(0)

# Random-feature regression -> reproduce DOUBLE DESCENT (peak at D = n).
d, n_train, n_test, noise_std = 20, 60, 4000, 0.5     # threshold at D = n = 60

Xtr = np.random.randn(n_train, d)
Xte = np.random.randn(n_test, d)
beta = np.random.randn(d) / math.sqrt(d)

def teacher(X):
    return np.tanh(X @ beta) + 0.3 * (X @ beta)

ytr = teacher(Xtr) + noise_std * np.random.randn(n_train)   # NOISY train labels
yte = teacher(Xte)                                          # CLEAN test labels

Dmax = 600
Wbank = np.random.randn(Dmax, d) / math.sqrt(d)

def feats(X, D):
    return np.maximum(X @ Wbank[:D].T, 0.0)                 # phi(x) = ReLU(W x)

### Step 5 — Sweep again and report the peak

Re-run the sweep, collecting train and test MSE into lists, then print where the test error peaks. The peak should land exactly at `D = n = 60` — the interpolation threshold — confirming the double-descent shape numerically.

In [ ]:
Ds = [2,5,10,15,20,30,40,48,54,58,60,62,66,72,84,100,130,180,250,350,500,600]
train, test = [], []
for D in Ds:
    Ptr = feats(Xtr, D)
    Pte = feats(Xte, D)
    theta = np.linalg.pinv(Ptr) @ ytr                       # min-norm least squares
    train.append(round(float(np.mean((Ptr @ theta - ytr)**2)), 4))
    test.append( round(float(np.mean((Pte @ theta - yte)**2)), 4))

print("D    :", Ds)
print("train:", train)
print("test :", test)
print("peak test MSE = %.3f at D = %d  (= n = %d, the interpolation threshold)"
      % (max(test), Ds[test.index(max(test))], n_train))
# train: [0.9437,0.8756,0.6127,0.4781,0.3883,0.2424,0.1678,0.121,0.0897,0.0034,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0]
# test : [0.799,0.7526,0.7554,0.7231,0.7561,1.0672,0.7846,1.3292,3.3675,24.9314,48.8568,19.3551,3.0518,2.0004,1.069,0.754,0.5743,0.506,0.3637,0.2645,0.2307,0.2183]
# peak test MSE = 48.857 at D = 60  (= n = 60, the interpolation threshold)
# Train MSE -> 0 at D=60 and stays 0; test MSE peaks at D=60 then falls again to 0.218.
# Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The label-noise ablation. You have a working double-descent sweep with noisy training labels
            and a sharp test-error peak at $D = n$. Set the training-label noise to zero, rerun, everything else
            identical. What do you expect to happen to (a) the peak and (b) the over-parameterized arm, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Find the noise term: training labels are y = teacher(x) + noise_std*randn(...). Set noise_std = 0. — _With clean labels, the teacher signal is the only thing to fit; there is no corrupted point that forces the model to contort._
- Reason about the threshold $D = n$: the minimum-norm interpolant now passes through clean, consistent points, so it stays close to the smooth teacher even when forced to interpolate. — _The peak's height is driven by how violently the fit must swing to hit wrong labels exactly. Remove the wrong labels and the swing &mdash; the variance explosion &mdash; mostly vanishes._
- Predict the curves: the peak shrinks to a small bump or disappears; the over-parameterized arm is roughly flat and low throughout. — _Both descents persist (more features still help), but without noise there is little variance to explode at the threshold, so the dramatic spike is gone._

**Answer:** The peak collapses. With clean labels the model never has to fit a contradictory point,
                 so the variance explosion at $D = n$ &mdash; the source of the spike &mdash; largely disappears.
                 You are left with a curve that descends and then stays low, much closer to a monotone "bigger is
                 a bit better" shape. This is the key diagnostic: double descent's spike is the cost of
                 interpolating noise. The over-parameterized arm still generalizes well because the
                 minimum-norm fit of clean, consistent data is just the smooth teacher.

</details>

**Problem 2.** Classical bias-variance theory predicts a single U-shaped test-error curve: one sweet spot, and
            "bigger is worse" past it. Double descent shows test error falling again for very large
            models. Does this mean the bias-variance tradeoff is wrong? Explain precisely where it holds
            and where it is incomplete.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Locate the U-curve on the double-descent plot: it is the under-parameterized region, $D \lt n$. — _Bias falls and variance rises with $D$ there, summing to a U &mdash; exactly the classical prediction. The first descent and the rise into the peak are the U._
- Identify what the classical theory left out: the regime $D \gt n$, where the model fits the training set exactly and the procedure picks the minimum-norm interpolant. — _Classical analysis never modeled deliberately interpolating the data, so it had no account of what happens past the threshold._
- Reason about variance past the threshold: as $D$ grows, the smoothest interpolant gets smoother, so variance falls again &mdash; the second descent. — _An implicit smoothness preference (minimum norm) tames variance once there is slack, which the bias-variance bound did not capture._

**Answer:** It is not wrong &mdash; it is incomplete. The bias-variance U-curve is exactly right in
                 the under-parameterized regime ($D \lt n$): that is the first descent and the climb to
                 the peak. What classical theory omitted is everything past the interpolation threshold,
                 because it never considered fitting the training set exactly on purpose. In the
                 over-parameterized regime the procedure picks the minimum-norm interpolant, whose variance
                 falls as the model grows &mdash; producing the second descent. Double descent is the
                 bias-variance curve continued past $D = n$. The U is the left half of a larger picture.

</details>

**Problem 3.** In the worked example you fit $\hat y = a + b x$ to points $x = [0,1,2]$, $y = [1,1,3]$ and got
            $\theta = (0.6667, 1)$ with training mean-squared error $0.2222$ &mdash; non-zero, because
            $D = 2 \lt n = 3$. Now add a third feature $x^2$ so $D = 3 = n$. Without solving the full system,
            argue what the training error becomes and why this point is special.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count: with $D = 3$ features and $n = 3$ points, the feature matrix $\Phi$ is $3 \times 3$ and (for distinct $x$) invertible. — _A square invertible system $\Phi\theta = y$ has an exact solution $\theta = \Phi^{-1}y$ &mdash; there is no residual left over._
- Conclude the fit passes through all three points exactly: $\hat y = y$, so every residual is $0$. — _Exact interpolation means zero training error by definition._
- Name the point: $D = n$ is the interpolation threshold &mdash; the model is just barely able to fit the training set. — _Below it, training error is positive (under-parameterized); at and above it, training error is zero. This is exactly where the test-error peak sits._

**Answer:** Training error becomes $0$. With $D = 3$ features and $n = 3$ distinct points the
                 feature matrix is square and invertible, so $\Phi\theta = y$ is solved exactly and the fit
                 passes through every training point &mdash; no residual. This $D = n$ point is the
                 interpolation threshold: training error first hits zero here, and it is precisely where
                 double descent's test-error peak appears. Below it ($D = 2$) we had positive training error
                 $0.2222$; at it, zero; and only by going well past it does test error come back down.

</details>